# Phase 3 : Fine-Tuning DistilBERT pour la Détection de Jailbreak/Prompt Injection
**Auteur :** Ayoub El Kharraf (Groupe 7 — Master M2I Cybersécurité CEH)
**Modèle :** `distilbert-base-uncased`
**Dataset :** `neuralchemy/Prompt-injection-dataset` (Configuration `full`)

## Étape 1 : Vérification du GPU (T4 / V100 / A100)
Assurez-vous d'avoir activé le GPU dans Colab : **Exécution > Modifier le type d'exécution > T4 GPU**.

In [ ]:
!nvidia-smi

## Étape 2 : Installation des dépendances

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas pyarrow matplotlib seaborn

## Étape 3 : Téléchargement ou import des données préparées
Si vous avez cloné votre dépôt GitHub, les données se trouvent dans `data_processed/full/`.
Sinon, exécutez la cellule suivante pour cloner le repo ou charger directement les fichiers.

In [ ]:
# Option A : Cloner votre dépôt (remplacer par votre URL si besoin)
# !git clone <VOTRE_REPO_URL>
# %cd llm-jailbreak-detection-benchmark

# Option B : Télécharger directement le dataset neuralchemy depuis Hugging Face
from datasets import load_dataset
import pandas as pd

print("[*] Chargement du dataset neuralchemy/Prompt-injection-dataset (full)...")
ds_full = load_dataset("neuralchemy/Prompt-injection-dataset", "full", revision="7d70432dfcf47a821612cbf9d34e9d9e3ad20e75")

train_df = ds_full["train"].to_pandas()
val_df = ds_full["validation"].to_pandas()
test_df = ds_full["test"].to_pandas()

# Nettoyage des 102 doublons exacts identifiés en Phase 1
duplicated = train_df[train_df.duplicated(subset=["text"], keep=False)]
duplicated_texts = duplicated["text"].unique()
mask_to_remove = (train_df["augmented"] == True) & (train_df["text"].isin(duplicated_texts))
train_df = train_df.loc[~mask_to_remove].copy()

# Ajout d'un example_id stable pour l'alignement test
test_df["example_id"] = [f"test_{i:04d}" for i in range(len(test_df))]

print(f"Train nettoyé : {len(train_df)} (102 doublons retirés)")
print(f"Validation    : {len(val_df)}")
print(f"Test          : {len(test_df)}")

## Étape 4 : Tokenisation (DistilBERT)

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], max_length=MAX_LENGTH, truncation=True)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label", "example_id"]]).map(tokenize_function, batched=True)

print("[*] Tokenisation terminée avec succès !")

## Étape 5 : Entraînement avec Hugging Face Trainer

In [ ]:
import time
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, set_seed
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(labels, preds)
    try:
        roc_auc = roc_auc_score(labels, probs)
    except Exception:
        roc_auc = 0.0
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "roc_auc": roc_auc}

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

start_t = time.time()
trainer.train()
duration_min = (time.time() - start_t) / 60
print(f"[*] Entraînement terminé en {duration_min:.2f} minutes !")

## Étape 6 : Évaluation sur le jeu de test et export des résultats

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

test_predictions = trainer.predict(test_ds)
logits = test_predictions.predictions
y_pred = np.argmax(logits, axis=1)
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
y_true = np.array(test_df["label"].tolist())

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
roc_auc = roc_auc_score(y_true, probs)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

metrics = {
    "model": "DistilBERT (distilbert-base-uncased)",
    "dataset": "neuralchemy/Prompt-injection-dataset (full)",
    "split_evaluated": "test",
    "test_samples": len(y_true),
    "accuracy": round(float(acc), 4),
    "precision_malicious": round(float(precision), 4),
    "recall_malicious": round(float(recall), 4),
    "f1_malicious": round(float(f1), 4),
    "macro_f1": round(float(f1_m), 4),
    "roc_auc": round(float(roc_auc), 4),
    "confusion_matrix": {
        "true_negative_benign": int(tn),
        "false_positive_benign_as_malicious": int(fp),
        "false_negative_malicious_missed": int(fn),
        "true_positive_malicious_detected": int(tp)
    },
    "training_time_minutes": round(duration_min, 2)
}

print("=== RÉSULTATS TEST DISTILBERT ===")
print(json.dumps(metrics, indent=2))

with open("distilbert_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

pred_df = pd.DataFrame({
    "example_id": test_df["example_id"],
    "y_true": y_true,
    "y_pred": y_pred,
    "score_malicious": probs
})
pred_df.to_csv("distilbert_test_predictions.csv", index=False)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Bénin (0)", "Malveillant (1)"], yticklabels=["Bénin (0)", "Malveillant (1)"])
plt.xlabel("Classe Prédite")
plt.ylabel("Classe Réelle")
plt.title("Matrice de Confusion — DistilBERT (Test)")
plt.savefig("distilbert_confusion_matrix.png", dpi=300)
plt.close()
print("[*] Fichiers générés : distilbert_test_predictions.csv, distilbert_metrics.json, distilbert_confusion_matrix.png")

## Étape 7 : Télécharger les fichiers de résultats vers votre PC

In [ ]:
from google.colab import files
files.download("distilbert_test_predictions.csv")
files.download("distilbert_metrics.json")
files.download("distilbert_confusion_matrix.png")